# Country Language Analyzer
This notebook utilizes `hanzipy` to analyze the Chinese names of countries grouped by geographical region. 

**Goal:** Create a detailed breakdown of each country name, showing the Pinyin and the dictionary definition for every individual character.


In [ ]:
import json
import re
import pandas as pd
from IPython.display import display, Markdown

# Initialize the hanzipy dictionary for character lookups
from hanzipy.dictionary import HanziDictionary
dictionary = HanziDictionary()

print("Setup complete. Dictionary loaded.")


## Data Loading
In this step, we load the country data from a local JSON file. The data is structured by region (Europe, Americas, Asia) and contains both the English and Chinese names for each country.


In [ ]:
# Load the data from the attached JSON file
try:
    with open('country_data.json', 'r', encoding='utf-8') as f:
        geo_data = json.load(f)
    print("Successfully loaded country_data.json")
except FileNotFoundError:
    print("Error: country_data.json not found. Please ensure the file is in the same directory.")


## Analysis and Table Generation
The following logic processes the data:
1. **Dynamic Column Sizing**: It calculates the longest Chinese name in the region to ensure the Markdown table has enough columns (P1, B1, P2, B2...).
2. **Character Extraction**: It iterates through each character of the country name to find its specific Pinyin and Meaning.
3. **Data Cleaning**: It uses Regular Expressions (`re`) to remove bracketed pinyin (e.g., `[Ying1 guo2]`) from the dictionary definitions to keep the table clean.
4. **Markdown Rendering**: It constructs a pipe-delimited string that Jupyter renders as a visual table.


In [21]:
import re

# Expanded list of keywords to filter out
FILTER_KEYWORDS = ['surname', 'abbr.', 'abbrev.', 'name of', 'clan', 'variant']

for region, region_countries in geo_data.items():
    display(Markdown(f"### Analysis for Region: {region}"))
    
    max_chars = max([len(item['zh']) for item in region_countries])
    
    headers = ["English Name", "Hanzi"]
    for i in range(1, max_chars + 1):
        headers.append(f"P{i}")
        headers.append(f"B{i}")
    
    header_row = "| " + " | ".join(headers) + " |"
    separator_row = "| " + " | ".join(["---"] * len(headers)) + " |"
    
    body_rows = []
    
    for item in region_countries:
        english_name = item['en']
        chinese_name = item['zh']
        row_data = [english_name, chinese_name]
        
        for char in chinese_name:
            lookup = dictionary.definition_lookup(char)
            if lookup:
                pinyin = lookup[0].get("pinyin", "N/A")
                
                best_meaning = None
                for entry in lookup:
                    defn = entry.get("definition", "")
                    
                    # --- STRONGER FILTERING LOGIC ---
                    # 1. Check for forbidden keywords
                    has_keyword = any(kw in defn.lower() for kw in FILTER_KEYWORDS)
                    
                    # 2. Check if the definition starts with a Chinese character 
                    # (This is where '个[ge4]' usually comes from)
                    starts_with_hanzi = any('\u4e00' <= c <= '\u9fff' for c in defn[:2])
                    
                    if not has_keyword and not starts_with_hanzi:
                        best_meaning = defn
                        break 
                
                # Fallback to the first entry if nothing passed the filter
                if not best_meaning:
                    best_meaning = lookup[0].get("definition", "unknown")
                
                # Clean bracketed pinyin and strip whitespace
                clean_meaning = re.sub(r'\[.*?\]', '', best_meaning).strip()
                
                row_data.append(pinyin)
                row_data.append(clean_meaning)
            else:
                row_data.append("N/A")
                row_data.append("unknown")
        
        while len(row_data) < (2 + (max_chars * 2)):
            row_data.append("-")
            
        body_rows.append("| " + " | ".join(row_data) + " |")

    md_table = "\n".join([header_row, separator_row] + body_rows)
    display(Markdown(md_table))


### Analysis for Region: Europe

| English Name | Hanzi | P1 | B1 | P2 | B2 | P3 | B3 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| United Kingdom | 英国 | Ying1 | hero/outstanding/excellent/(literary) flower/blossom | Guo2 | country/nation/state/national/CL:個|个 | - | - |
| Ireland | 爱尔兰 | ai4 | to love/to be fond of/to like/affection/to be inclined (to do sth)/to tend to (happen) | er3 | thus/so/like that/you/thou | Lan2 | orchid (蘭花|兰花 Cymbidium goeringii)/fragrant thoroughwort (蘭草|兰草 Eupatorium fortunei)/lily magnolia (木蘭|木兰) |
| France | 法国 | fa3 | variant of 法 | Guo2 | country/nation/state/national/CL:個|个 | - | - |
| Spain | 西班牙 | Xi1 | west | Ban1 | team/class/squad/work shift/ranking/CL:個|个/classifier for groups of people and scheduled transport vehicles | ya2 | tooth/ivory/CL:顆|颗 |
| Germany | 德国 | De2 | virtue/goodness/morality/ethics/kindness/favor/character/kind | Guo2 | country/nation/state/national/CL:個|个 | - | - |
| Italy | 意大利 | Yi4 | idea/meaning/thought/to think/wish/desire/intention/to expect/to anticipate | da4 | big/huge/large/major/great/wide/deep/older (than)/oldest/eldest/greatly/very much/(dialect) father/father's elder or younger brother | Li4 | sharp/favorable/advantage/benefit/profit/interest/to do good to/to benefit |
| Belgium | 比利时 | Bi3 | to compare/(followed by a noun and adjective) more {adj.} than {noun}/ratio/to gesture/(Taiwan pr.  in some compounds derived from Classical Chinese) | Li4 | sharp/favorable/advantage/benefit/profit/interest/to do good to/to benefit | shi2 | o'clock/time/when/hour/season/period |
| Netherlands | 荷兰 | He2 | lotus | Lan2 | orchid (蘭花|兰花 Cymbidium goeringii)/fragrant thoroughwort (蘭草|兰草 Eupatorium fortunei)/lily magnolia (木蘭|木兰) | - | - |
| Switzerland | 瑞士 | rui4 | lucky/auspicious/propitious/rayl (acoustical unit) | Shi4 | member of the senior ministerial class (old)/scholar (old)/bachelor/honorific/soldier/noncommissioned officer/specialist worker | - | - |
| Sweden | 瑞典 | rui4 | lucky/auspicious/propitious/rayl (acoustical unit) | dian3 | canon/law/standard work of scholarship/literary quotation or allusion/ceremony/to be in charge of/to mortgage or pawn | - | - |
| Norway | 挪威 | nuo2 | to shift/to move | wei1 | power/might/prestige | - | - |
| Denmark | 丹麦 | dan1 | red/pellet/powder/cinnabar | Mai4 | wheat/barley/oats | - | - |
| Poland | 波兰 | Bo1 | wave/ripple/storm/surge | Lan2 | orchid (蘭花|兰花 Cymbidium goeringii)/fragrant thoroughwort (蘭草|兰草 Eupatorium fortunei)/lily magnolia (木蘭|木兰) | - | - |
| Ukraine | 乌克兰 | Wu1 | crow/black | Ke4 | to be able to/to subdue/to restrain/to overcome/gram/Tibetan unit of land area, about 6 ares | Lan2 | orchid (蘭花|兰花 Cymbidium goeringii)/fragrant thoroughwort (蘭草|兰草 Eupatorium fortunei)/lily magnolia (木蘭|木兰) |
| Russia | 俄罗斯 | E2 | suddenly/very soon | Luo2 | gauze/to collect/to gather/to catch/to sift | Si1 | (phonetic)/this |
| Greece | 希腊 | xi1 | to hope/to admire/variant of 稀 | xi1 | dried meat/also pr. | - | - |

### Analysis for Region: Americas

| English Name | Hanzi | P1 | B1 | P2 | B2 | P3 | B3 | P4 | B4 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| United States | 美国 | Mei3 | beautiful/very satisfactory/good/to beautify/to be pleased with oneself | Guo2 | country/nation/state/national/CL:個|个 | - | - | - | - |
| Canada | 加拿大 | Jia1 | to add/plus/(used after an adverb such as 不, 大, 稍 etc, and before a disyllabic verb, to indicate that the action of the verb is applied to sth or sb previously mentioned)/to apply (restrictions etc) to (sb)/to give (support, consideration etc) to (sth) | na2 | to hold/to seize/to catch/to apprehend/to take/(used in the same way as 把: to mark the following noun as a direct object) | da4 | big/huge/large/major/great/wide/deep/older (than)/oldest/eldest/greatly/very much/(dialect) father/father's elder or younger brother | - | - |
| Mexico | 墨西哥 | Mo4 | ink stick/China ink/CL:塊|块/corporal punishment consisting of tattooing characters on the victim's forehead | Xi1 | west | ge1 | elder brother | - | - |
| Brazil | 巴西 | Ba1 | Ba state during Zhou dynasty (in east of modern Sichuan)/abbr. for east Sichuan or Chongqing/surname Ba/abbr. for Palestine or Palestinian/abbr. for Pakistan | Xi1 | west | - | - | - | - |
| Argentina | 阿根廷 | A1 | prefix used before monosyllabic names, kinship terms etc to indicate familiarity/used in transliteration/also pr. | gen1 | root/basis/classifier for long slender objects, e.g. cigarettes, guitar strings/CL:條|条/radical (chemistry) | ting2 | palace courtyard | - | - |
| Colombia | 哥伦比亚 | ge1 | elder brother | lun2 | human relationship/order/coherence | Bi3 | to compare/(followed by a noun and adjective) more {adj.} than {noun}/ratio/to gesture/(Taiwan pr.  in some compounds derived from Classical Chinese) | Ya4 | Asia/Asian/Taiwan pr. |
| Peru | 秘鲁 | mi4 | see 秘魯|秘鲁 | Lu3 | (bound form) crass/stupid/rude/(used to represent the sounds of "ru", "lu" etc in loanwords) | - | - | - | - |
| Chile | 智利 | zhi4 | wisdom/knowledge | Li4 | sharp/favorable/advantage/benefit/profit/interest/to do good to/to benefit | - | - | - | - |
| Venezuela | 委内瑞拉 | Wei3 | same as 逶 in 逶迤 winding, curved | nei4 | inside/inner/internal/within/interior | rui4 | lucky/auspicious/propitious/rayl (acoustical unit) | la1 | to pull/to play (a bowed instrument)/to drag/to draw/to chat/(coll.) to empty one's bowels |
| Cuba | 古巴 | Gu3 | ancient/old/paleo- | Ba1 | Ba state during Zhou dynasty (in east of modern Sichuan)/abbr. for east Sichuan or Chongqing/surname Ba/abbr. for Palestine or Palestinian/abbr. for Pakistan | - | - | - | - |
| Jamaica | 牙买加 | ya2 | tooth/ivory/CL:顆|颗 | mai3 | to buy/to purchase | Jia1 | to add/plus/(used after an adverb such as 不, 大, 稍 etc, and before a disyllabic verb, to indicate that the action of the verb is applied to sth or sb previously mentioned)/to apply (restrictions etc) to (sb)/to give (support, consideration etc) to (sth) | - | - |
| Haiti | 海地 | Hai3 | ocean/sea/CL:個|个,片/great number of people or things/(dialect) numerous | de5 | -ly/structural particle: used before a verb or adjective, linking it to preceding modifying adverbial adjunct | - | - | - | - |
| Panama | 巴拿马 | Ba1 | Ba state during Zhou dynasty (in east of modern Sichuan)/abbr. for east Sichuan or Chongqing/surname Ba/abbr. for Palestine or Palestinian/abbr. for Pakistan | na2 | to hold/to seize/to catch/to apprehend/to take/(used in the same way as 把: to mark the following noun as a direct object) | Ma3 | horse/CL:匹/horse or cavalry piece in Chinese chess/knight in Western chess | - | - |

### Analysis for Region: Asia

| English Name | Hanzi | P1 | B1 | P2 | B2 | P3 | B3 | P4 | B4 | P5 | B5 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| China | 中国 | Zhong1 | within/among/in/middle/center/while (doing sth)/during/(dialect) OK/all right | Guo2 | country/nation/state/national/CL:個|个 | - | - | - | - | - | - |
| Japan | 日本 | Ri4 | sun/day/date, day of the month | ben3 | root/stem/origin/source/this/the current/original/inherent/originally/classifier for books, periodicals, files etc | - | - | - | - | - | - |
| South Korea | 韩国 | Han2 | Han, one of the Seven Hero States of the Warring States 戰國七雄|战国七雄/Korea from the fall of the Joseon dynasty in 1897/Korea, esp. South Korea 大韓民國|大韩民国/surname Han | Guo2 | country/nation/state/national/CL:個|个 | - | - | - | - | - | - |
| North Korea | 朝鲜 | Chao2 | imperial or royal court/government/dynasty/reign of a sovereign or emperor/court or assembly held by a sovereign or emperor/to make a pilgrimage to/facing/towards | xian3 | fresh/bright (in color)/delicious/tasty/delicacy/aquatic foods | - | - | - | - | - | - |
| India | 印度 | Yin4 | to print/to mark/to engrave/a seal/a print/a stamp/a mark/a trace/image | du4 | to pass/to spend (time)/measure/limit/extent/degree of intensity/degree (angles, temperature etc)/kilowatt-hour/classifier for events and occurrences | - | - | - | - | - | - |
| Singapore | 新加坡 | Xin1 | new/newly/meso- (chemistry) | Jia1 | to add/plus/(used after an adverb such as 不, 大, 稍 etc, and before a disyllabic verb, to indicate that the action of the verb is applied to sth or sb previously mentioned)/to apply (restrictions etc) to (sb)/to give (support, consideration etc) to (sth) | po1 | slope/CL:個|个/sloping/slanted | - | - | - | - |
| Thailand | 泰国 | Tai4 | safe/peaceful/most/grand | Guo2 | country/nation/state/national/CL:個|个 | - | - | - | - | - | - |
| Vietnam | 越南 | Yue4 | to exceed/to climb over/to surpass/the more... the more | Nan2 | south | - | - | - | - | - | - |
| Indonesia | 印度尼西亚 | Yin4 | to print/to mark/to engrave/a seal/a print/a stamp/a mark/a trace/image | du4 | to pass/to spend (time)/measure/limit/extent/degree of intensity/degree (angles, temperature etc)/kilowatt-hour/classifier for events and occurrences | ni2 | Buddhist nun/(often used in phonetic spellings) | Xi1 | west | Ya4 | Asia/Asian/Taiwan pr. |
| Malaysia | 马来西亚 | Ma3 | horse/CL:匹/horse or cavalry piece in Chinese chess/knight in Western chess | lai2 | to come/to arrive/to come round/ever since/next | Xi1 | west | Ya4 | Asia/Asian/Taiwan pr. | - | - |
| Philippines | 菲律宾 | Fei1 | luxuriant (plant growth)/rich with fragrance/phenanthrene C14H10 | Lu:4 | law | bin1 | visitor/guest/object (in grammar) | - | - | - | - |
| Saudi Arabia | 沙特阿拉伯 | Sha1 | surname Sha | te4 | special/unique/distinguished/especially/unusual/very/abbr. for 特克斯, tex | A1 | prefix used before monosyllabic names, kinship terms etc to indicate familiarity/used in transliteration/also pr. | la1 | to pull/to play (a bowed instrument)/to drag/to draw/to chat/(coll.) to empty one's bowels | ba4 | one hundred (old) |
| UAE | 阿联酋 | A1 | prefix used before monosyllabic names, kinship terms etc to indicate familiarity/used in transliteration/also pr. | lian2 | to ally/to unite/to join/(poetry) antithetical couplet | qiu2 | tribal chief | - | - | - | - |
| Iran | 伊朗 | Yi1 | (old) third person singular pronoun ("he" or "she")/second person singular pronoun ("you")/(May 4th period) third person singular feminine pronoun ("she")/(Classical Chinese) introductory particle with no specific meaning/that (preceding a noun) | lang3 | clear/bright | - | - | - | - | - | - |
| Turkey | 土耳其 | Tu3 | earth/dust/clay/local/indigenous/crude opium/unsophisticated/one of the eight categories of ancient musical instruments 八音 | er3 | ear/handle (archaeology)/and that is all (Classical Chinese) | qi2 | his/her/its/their/that/such/it (refers to sth preceding it) | - | - | - | - |